# Phase 7 — Stage 2 retrain (FROZEN dual-path Stage 1)

**Goal**: recover diffusion skill (Pearson > 0.82) while preserving causality
(mu_HR ablation ~99%). Stage 1 (encoder + RCN + regression_head + dual_path) is
FROZEN; only the diffusion decoder is trained.

**Modes**:
- `SMOKE_MODE = True` (default): 3 epochs warm-start + per-epoch BS30 eval +
  GO/NO-GO decision. Cost ~2-3h on T4.
- `SMOKE_MODE = False`: 200 epochs (matches non-causal training), eval every 10
  epochs. Cost ~28h training + ~8h eval on T4.

**Init**:
- Warm-start: load Stage 2 weights from `oracle_9node/seed_42/epoch_last.pth`
  (existing Stage 2 trained on mu_A only).
- From scratch: set `FROM_SCRATCH = True` — diffusion weights randomly init'd.

**Loss**: EDM Karras (Karras 2022 §3) MSE on residual
`= HR_log1p - baseline_log - mu_total` via
`diffusion_decoder.compute_loss_edm(...)`. Standard recipe:
- Adam lr=1e-4, betas=(0.9, 0.999), wd=0
- gradient clip 1.0
- LinearLR warmup (1000 steps)
- EMA decay 0.9999 on diffusion params
- AMP fp16 on CUDA, fp32 fallback otherwise

**Output**: `oracle_9node/seed_42/phase7_stage2_retrain/`
- `epoch_last.pth` (resume target)
- `epoch_best.pth` (best Pearson)
- `final_validation_metrics.json` (BS30 protocol, Phase 5/6 format) — full mode only


In [ ]:
# === Cell 1 : Bootstrap Colab ===
import subprocess, shlex, os, sys
from pathlib import Path

REPO_DIR   = Path('/content/climate_data')
GIT_URL    = 'https://github.com/leonelkenfack/stcdgm.git'
GIT_BRANCH = 'four-node-causal'

if not (REPO_DIR / '.git').exists():
    subprocess.run(shlex.split(
        f'git clone --depth 200 -b {GIT_BRANCH} {GIT_URL} {REPO_DIR}'), check=True)
else:
    subprocess.run(shlex.split(
        f'git -C {REPO_DIR} fetch --depth=200 origin {GIT_BRANCH}'), check=True)
    subprocess.run(shlex.split(
        f'git -C {REPO_DIR} reset --hard origin/{GIT_BRANCH}'), check=True)

os.chdir(str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / 'src'))

try:
    import torch_geometric; import cftime; import h5netcdf; import xbatcher; import diffusers
    from omegaconf import OmegaConf
except ImportError:
    EXTRA_DEPS = [
        'torch_geometric', 'omegaconf==2.3.0', 'hydra-core==1.3.2',
        'diffusers==0.36.0', 'einops', 'scipy', 'h5py', 'netCDF4',
        'xarray', 'dask', 'zarr', 'safetensors==0.7.0',
        'xbatcher', 'webdataset', 'cftime', 'h5netcdf',
    ]
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + EXTRA_DEPS, check=True)

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ModuleNotFoundError:
    print('[bootstrap] not on Colab - Drive mount skipped')

print(f'[bootstrap] cwd={os.getcwd()}  branch={GIT_BRANCH}')


In [ ]:
# === Cell 2 : Constants ===
import json
import numpy as np
import torch
from pathlib import Path
from omegaconf import OmegaConf

# --- Mode flags ---
SMOKE_MODE   = True   # set False for full 200-epoch retrain
SMOKE_EPOCHS = 3
FULL_EPOCHS  = 200

# Warm-start vs from scratch
FROM_SCRATCH = False  # True = init Stage 2 weights randomly (cold)

# --- Drive paths ---
DRIVE_ROOT       = Path('/content/drive/MyDrive/climate_data')
ORACLE_9N        = DRIVE_ROOT / 'oracle_9node' / 'seed_42'
CKPT_DUALPATH    = ORACLE_9N / 'epoch_best_dualpath.pth'
CKPT_STAGE2_INIT = ORACLE_9N / 'epoch_last.pth'   # warm-start source (mu_A-only Stage 2)
SIGMA_DATA_NEW   = 0.193                          # phase6 dualpath recalibrated sigma_data

# Output dir for retrain artefacts
RETRAIN_DIR = ORACLE_9N / 'phase7_stage2_retrain'
RETRAIN_DIR.mkdir(parents=True, exist_ok=True)
CKPT_LAST   = RETRAIN_DIR / 'epoch_last.pth'
CKPT_BEST   = RETRAIN_DIR / 'epoch_best.pth'
HISTORY_JSON= RETRAIN_DIR / 'training_history.json'

# --- Training hyperparams (mirror non-causal Stage 2) ---
LR                = 1e-4         # Adam default for Stage 2 retrain
BETA1, BETA2      = 0.9, 0.999
WEIGHT_DECAY      = 0.0
GRADIENT_CLIP     = 1.0
EMA_DECAY         = 0.9999       # standard for diffusion (Karras EDM2)
WARMUP_STEPS      = 1000         # LinearLR warmup
USE_AMP           = True         # mixed precision OK for Stage 2 (well-conditioned)
SCALER_INIT_SCALE = 2 ** 13

# --- Eval hyperparams (BS30 protocol, identical to phase6 final_validation) ---
EVAL_EVERY_N_EPOCHS = 1 if SMOKE_MODE else 10  # smoke: every epoch; full: every 10
EVAL_N_BATCHES      = 16
EVAL_K_SAMPLES      = 64
EVAL_N_STEPS_DIFF   = 18
EVAL_N_INTERVENTION = 4

# --- Resume ---
RESUME = True  # auto-resume if CKPT_LAST exists in RETRAIN_DIR

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

EPOCHS_TARGET = SMOKE_EPOCHS if SMOKE_MODE else FULL_EPOCHS
print(f'[Cell 2] DEVICE             = {DEVICE}')
print(f'[Cell 2] SMOKE_MODE         = {SMOKE_MODE}  (epochs={EPOCHS_TARGET})')
print(f'[Cell 2] FROM_SCRATCH       = {FROM_SCRATCH}')
print(f'[Cell 2] CKPT_DUALPATH      = {CKPT_DUALPATH}  exists={CKPT_DUALPATH.exists()}')
print(f'[Cell 2] CKPT_STAGE2_INIT   = {CKPT_STAGE2_INIT}  exists={CKPT_STAGE2_INIT.exists()}')
print(f'[Cell 2] RETRAIN_DIR        = {RETRAIN_DIR}')
print(f'[Cell 2] CKPT_LAST exists   = {CKPT_LAST.exists()}  (RESUME={RESUME})')
print(f'[Cell 2] LR={LR}  EMA_DECAY={EMA_DECAY}  GRAD_CLIP={GRADIENT_CLIP}')
print(f'[Cell 2] BS30 eval : N_BATCHES={EVAL_N_BATCHES} K={EVAL_K_SAMPLES} '
      f'STEPS={EVAL_N_STEPS_DIFF}  every {EVAL_EVERY_N_EPOCHS} epoch(s)')


In [ ]:
# === Cell 3 : Config + Pipeline + Dataloaders (9-node) ===
from torch.utils.data import DataLoader as _DataLoader, IterableDataset
from st_cdgm.data.pipeline import NetCDFDataPipeline
from st_cdgm.models.graph_builder import HeteroGraphBuilder
from path_c_plus.scripts.option_c_helpers import PATHCPLUS_HYPERPARAM_OVERRIDES
from path_c_plus.scripts.gpu_detect import detect_gpu_profile, print_profile_banner

# --- Config ---
CONFIG = OmegaConf.load('config/training_config.yaml')
_corrdiff = OmegaConf.load('config/training_config_corrdiff_normal.yaml')
CONFIG = OmegaConf.merge(CONFIG, _corrdiff)

EXTENDED_9NODE = True
GPU_PROFILE = detect_gpu_profile()
print_profile_banner(GPU_PROFILE)

CONFIG.training.batch_size  = 1
CONFIG.training.use_amp     = GPU_PROFILE['use_amp']
CONFIG.training.num_workers = GPU_PROFILE['num_workers']

ts_cfg = CONFIG.two_stage
ts_cfg.stage1['lambda_dag_prior'] = PATHCPLUS_HYPERPARAM_OVERRIDES['lambda_dag_prior']
ts_cfg.stage1['g_phys_alpha']     = PATHCPLUS_HYPERPARAM_OVERRIDES['g_phys_alpha']

OmegaConf.set_struct(CONFIG, False)
_existing_mp = {m.name for m in CONFIG.encoder.metapaths}
for _m in [
    {'name': 'Q850', 'src': 'Q850', 'relation': 'causes', 'target': 'GP850', 'pool': 'mean'},
    {'name': 'W500', 'src': 'W500', 'relation': 'causes', 'target': 'GP500', 'pool': 'mean'},
    {'name': 'IVT',  'src': 'IVT',  'relation': 'causes', 'target': 'GP850', 'pool': 'mean'},
]:
    if _m['name'] not in _existing_mp:
        CONFIG.encoder.metapaths.append(OmegaConf.create(_m))
print(f'[Cell 3] metapaths -> {[m.name for m in CONFIG.encoder.metapaths]}')

K9_DATES = {
    'train':   ['1980-01-01', '2009-12-31'],
    'val':     ['2010-01-01', '2011-12-31'],
    'test':    ['2012-01-01', '2013-12-31'],
    'holdout': ['2014-01-01', '2014-12-31'],
}

_ON_COLAB  = 'google.colab' in sys.modules or Path('/content').exists()
DATA_ROOT  = Path('/content/drive/MyDrive/climate_data/data') if _ON_COLAB else Path('data/raw')
LR_PATH    = str(DATA_ROOT / 'train' / 'predictor_ACCESS-CM2_hist.nc')
HR_PATH    = str(DATA_ROOT / 'train' / 'pr_ACCESS-CM2_hist.nc')
_static_p  = DATA_ROOT / 'static_predictors' / 'ERA5_eval_ccam_12km.198110_NZ_Invariant.nc'
_mean_p    = DATA_ROOT / 'train' / 'means_ACCESS-CM2.nc'
_std_p     = DATA_ROOT / 'train' / 'stds_ACCESS-CM2.nc'
STATIC_PATH = str(_static_p) if _static_p.exists() else None
MEAN_PATH   = str(_mean_p)   if _mean_p.exists()   else None
STD_PATH    = str(_std_p)    if _std_p.exists()    else None

SEQ_LEN             = int(CONFIG.data.seq_len)
BASELINE_STRATEGY   = str(CONFIG.data.baseline_strategy)
BASELINE_FACTOR     = int(CONFIG.data.baseline_factor)
NORMALIZE           = bool(CONFIG.data.normalize)
PRECIPITATION_DELTA = float(CONFIG.data.precipitation_delta)
NAN_FILL_STRATEGY   = str(CONFIG.data.nan_fill_strategy)
_default_lr = ['q_500', 'q_850', 'u_500', 'u_850', 'v_500', 'v_850', 't_500', 't_850']
LR_VARIABLES  = list(CONFIG.data.lr_variables)  if CONFIG.data.get('lr_variables')  else _default_lr
HR_VARIABLES  = list(CONFIG.data.hr_variables)  if CONFIG.data.get('hr_variables')  else ['pr']
STATIC_VARIABLES = list(CONFIG.data.static_variables) if CONFIG.data.get('static_variables') else []

pipeline = NetCDFDataPipeline(
    lr_path=LR_PATH, hr_path=HR_PATH, static_path=STATIC_PATH,
    seq_len=SEQ_LEN, baseline_strategy=BASELINE_STRATEGY,
    baseline_factor=BASELINE_FACTOR, normalize=NORMALIZE,
    nan_fill_strategy=NAN_FILL_STRATEGY,
    precipitation_delta=PRECIPITATION_DELTA,
    lr_variables=LR_VARIABLES, hr_variables=HR_VARIABLES,
    static_variables=STATIC_VARIABLES,
    means_path=MEAN_PATH, stds_path=STD_PATH,
    train_start_date=K9_DATES['train'][0], train_end_date=K9_DATES['train'][1],
    val_start_date=K9_DATES['val'][0],     val_end_date=K9_DATES['val'][1],
    test_start_date=K9_DATES['test'][0],   test_end_date=K9_DATES['test'][1],
    temporal_holdout_start_date=K9_DATES['holdout'][0],
    temporal_holdout_end_date=K9_DATES['holdout'][1],
)
print('[Cell 3] Pipeline ready')

train_dataset = pipeline.build_sequence_dataset(split='train', seq_len=SEQ_LEN,
                                                 stride=int(CONFIG.data.stride), as_torch=True)
val_dataset   = pipeline.build_sequence_dataset(split='val',   seq_len=SEQ_LEN,
                                                 stride=int(CONFIG.data.stride), as_torch=True)

BATCH_SIZE  = 1
NUM_WORKERS = int(CONFIG.training.num_workers)
PIN_MEMORY  = bool(torch.cuda.is_available())
_loader_kw  = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                   pin_memory=PIN_MEMORY, collate_fn=lambda x: x)
if NUM_WORKERS > 0:
    _loader_kw.update(persistent_workers=True, prefetch_factor=2)

train_dataloader = _DataLoader(train_dataset,
                                shuffle=not isinstance(train_dataset, IterableDataset),
                                **_loader_kw)
val_dataloader   = _DataLoader(val_dataset, shuffle=False, **_loader_kw)

lr_shape = tuple(CONFIG.graph.lr_shape)
hr_shape = tuple(CONFIG.graph.hr_shape)
builder  = HeteroGraphBuilder(
    lr_shape=lr_shape, hr_shape=hr_shape,
    static_dataset=pipeline.get_static_dataset(),
    include_mid_layer=CONFIG.graph.include_mid_layer,
    extended_9node=EXTENDED_9NODE,
)
print(f'[Cell 3] Builder  lr_shape={lr_shape}  hr_shape={hr_shape}  dyn={builder.dynamic_node_types}')

_LR_VARS = list(CONFIG.data.lr_variables)
_VI = {v: i for i, v in enumerate(_LR_VARS)}
_Q_IDX = [_VI[v] for v in ('q_850', 'q_500', 'q_250') if v in _VI]
_W_IDX = [_VI[v] for v in ('w_850', 'w_500', 'w_250') if v in _VI]
_IVT_LEVELS = [lev for lev in ('850', '500', '250')
               if f'q_{lev}' in _VI and f'u_{lev}' in _VI and f'v_{lev}' in _VI]

def _compute_ivt_nodes(lr0):
    acc = None
    for lev in _IVT_LEVELS:
        q = lr0[:, [_VI[f'q_{lev}']]]
        u = lr0[:, [_VI[f'u_{lev}']]]
        v = lr0[:, [_VI[f'v_{lev}']]]
        term = q * torch.sqrt(u * u + v * v + 1e-12)
        acc = term if acc is None else acc + term
    if acc is None:
        acc = lr0[:, 0:1] * 0.0
    return acc / (len(_IVT_LEVELS) + 1e-8)

def _ensure_2d(t):
    return t.unsqueeze(-1) if t.dim() == 1 else t

def convert_sample_to_batch(sample, builder, device):
    lr_seq = sample['lr']
    seq_len = lr_seq.shape[0]
    lr_nodes_steps = [builder.lr_grid_to_nodes(lr_seq[t]) for t in range(seq_len)]
    lr_tensor = torch.stack(lr_nodes_steps, dim=0)
    lr0 = lr_nodes_steps[0]
    if EXTENDED_9NODE:
        _ivt = _compute_ivt_nodes(lr0)
        dynamic_features = {}
        for nt in builder.dynamic_node_types:
            if nt == 'Q850':   dynamic_features[nt] = _ensure_2d(lr0[:, _Q_IDX] if _Q_IDX else lr0)
            elif nt == 'W500': dynamic_features[nt] = _ensure_2d(lr0[:, _W_IDX] if _W_IDX else lr0)
            elif nt == 'IVT':  dynamic_features[nt] = _ensure_2d(_ivt)
            else:              dynamic_features[nt] = _ensure_2d(lr0)
    else:
        dynamic_features = {nt: _ensure_2d(lr0) for nt in builder.dynamic_node_types}
    hetero = builder.prepare_step_data(dynamic_features).to(device)
    return {
        'lr':       lr_tensor,
        'lr_grid':  lr_seq,
        'residual': sample['residual'],
        'baseline': sample.get('baseline'),
        'hetero':   hetero,
        'time':     sample.get('time'),
    }

def iterate_batches(dataloader, builder, device):
    for batch_list in dataloader:
        if not isinstance(batch_list, list):
            batch_list = [batch_list]
        yield [convert_sample_to_batch(s, builder, device) for s in batch_list]

_probe = next(iter(train_dataset))
C_LR = _probe['lr'].shape[1]
_n_train = len(train_dataset) if hasattr(train_dataset, '__len__') else '?'
_n_val   = len(val_dataset)   if hasattr(val_dataset,   '__len__') else '?'
print(f'[Cell 3] C_LR={C_LR}  lr_shape={lr_shape}  hr_shape={hr_shape}')
print(f'[Cell 3] train={_n_train} samples  val={_n_val} samples')

H_HR, W_HR = int(hr_shape[0]), int(hr_shape[1])
print(f'[Cell 3] HR shape = ({H_HR}, {W_HR})')


In [ ]:
# === Cell 4 : Load Stage 1 dual-path (encoder + RCN + head + dual_path) FROZEN ===
# Mirror phase6_dualpath_final_validation Cell 5. All Stage 1 parameters are
# permanently frozen here (requires_grad=False, eval()) — the gradient only
# touches the diffusion decoder built in Cell 5.
from st_cdgm.models.dual_path_stage1 import DualPathPredictor
from st_cdgm.training.stage1_paths import batch_lr_grid_last
from st_cdgm.models.intelligible_encoder import (
    IntelligibleVariableEncoder, IntelligibleVariableConfig,
)
from st_cdgm.models.causal_rcn import RCNCell, RCNSequenceRunner
from st_cdgm.models.regression_head import GraphToGridDecoder


def _parse_encoder_metapaths_from_ckpt(enc_sd):
    seen, order = {}, []
    for k in enc_sd:
        if not k.startswith('metapath_convs.'):
            continue
        rest = k[len('metapath_convs.'):]
        parts = rest.split('__')
        if len(parts) < 4:
            continue
        name = parts[0]; src = parts[1]; rel = parts[2]; tgt = parts[3].split('.')[0]
        if name not in seen:
            seen[name] = (src, rel, tgt); order.append(name)
    return [(n,) + seen[n] for n in order]


def _clean_sd(sd):
    if sd is None:
        return None
    if any('_orig_mod' in k for k in sd):
        sd = {k.replace('_orig_mod.', ''): v for k, v in sd.items()}
    return sd


def _safe_load(module, ck_data, keys, label):
    for key in keys:
        sd = ck_data.get(key)
        if sd is not None:
            sd = _clean_sd(sd)
            try:
                missing, unexpected = module.load_state_dict(sd, strict=False)
                msg = f'  [{label}] loaded from "{key}"'
                if missing:    msg += f'  | missing={len(missing)}'
                if unexpected: msg += f'  | unexpected={len(unexpected)}'
                print(msg)
                return True
            except Exception as e:
                print(f'  [{label}] FAILED with "{key}" : {type(e).__name__}: {e}')
                continue
    print(f'  [{label}] no valid key found in {keys}')
    return False


def _strip_prefixes(sd):
    if sd is None:
        return None
    prefixes = ['_orig_mod.', 'module.']
    out = {}
    for k, v in sd.items():
        nk = k
        for p in prefixes:
            if nk.startswith(p):
                nk = nk[len(p):]
        out[nk] = v
    return out


print(f'[Cell 4] Loading Stage 1 dual-path : {CKPT_DUALPATH}')
ck_s1 = torch.load(CKPT_DUALPATH, map_location=DEVICE, weights_only=False)
print(f'[Cell 4] ckpt keys[:12] = {sorted(ck_s1.keys())[:12]}')

enc_sd = _clean_sd(ck_s1.get('encoder_state_dict', {}))
parsed = _parse_encoder_metapaths_from_ckpt(enc_sd)
print(f'  metapaths detected : {[t[0] for t in parsed]}')
cfgs = [
    IntelligibleVariableConfig(name=n, meta_path=(s, r, t), pool='mean')
    for n, s, r, t in parsed
]
encoder = IntelligibleVariableEncoder(
    configs=cfgs,
    hidden_dim=int(CONFIG.encoder.hidden_dim),
    conditioning_dim=int(CONFIG.encoder.conditioning_dim),
).to(DEVICE)
n_vars = len(cfgs)
num_vars = n_vars  # exposed for Cell 5 (projection_class_embeddings_input_dim)

_probe_b = next(iter(val_dataset))
_lr_nodes = builder.lr_grid_to_nodes(_probe_b['lr'][0])
rcn_driver_dim = _lr_nodes.shape[-1]

rcn_cell = RCNCell(
    num_vars=n_vars,
    hidden_dim=int(CONFIG.rcn.hidden_dim),
    driver_dim=rcn_driver_dim,
    reconstruction_dim=rcn_driver_dim,
    dropout=float(CONFIG.rcn.dropout),
).to(DEVICE)
rcn_runner = RCNSequenceRunner(rcn_cell, detach_interval=CONFIG.rcn.get('detach_interval'))

rh_cfg = CONFIG.two_stage.regression_head
regression_head = GraphToGridDecoder(
    d_model=int(rh_cfg.d_model),
    hr_h=H_HR, hr_w=W_HR,
    intermediate_h=int(rh_cfg.intermediate_h),
    intermediate_w=int(rh_cfg.intermediate_w),
    n_heads=int(rh_cfg.n_heads),
    refine_channels=int(rh_cfg.refine_channels),
    output_channels=1,
).to(DEVICE)

PATH_B_KIND          = 'unet'
PATH_B_UNET_CHANNELS = (32, 64, 128)
PATH_B_UNET_LR_SHAPE = (23, 26)
PATH_B_BASE_CH       = 48
GATE_MAX_MEAN        = 0.40
dual_path = DualPathPredictor(
    in_channels=C_LR,
    base_ch=PATH_B_BASE_CH,
    hr_h=H_HR, hr_w=W_HR,
    gate_max_mean=GATE_MAX_MEAN,
    path_b_kind=PATH_B_KIND,
    path_b_unet_channels=PATH_B_UNET_CHANNELS,
    path_b_unet_lr_shape=PATH_B_UNET_LR_SHAPE,
).to(DEVICE)

_safe_load(encoder,         ck_s1, ['encoder_state_dict'],                            'encoder')
_safe_load(rcn_cell,        ck_s1, ['rcn_cell_state_dict', 'rcn_state_dict'],         'rcn_cell')
_safe_load(regression_head, ck_s1, ['regression_head_state_dict', 'head_state_dict'], 'regression_head')
_safe_load(dual_path,       ck_s1, ['dual_path_state_dict'],                          'dual_path')

# FREEZE all Stage 1 modules (no gradient flows here during retrain).
_stage1_n_total = 0
_stage1_n_trainable = 0
for m in [encoder, rcn_cell, regression_head, dual_path]:
    for p in m.parameters():
        p.requires_grad_(False)
        _stage1_n_total += p.numel()
    m.eval()
for m in [encoder, rcn_cell, regression_head, dual_path]:
    for p in m.parameters():
        if p.requires_grad:
            _stage1_n_trainable += p.numel()
assert _stage1_n_trainable == 0, (
    f'Stage 1 freeze failed : {_stage1_n_trainable} trainable params remain '
    f'(expected 0).'
)
print(f'  [verify] Stage 1 frozen : {_stage1_n_total:,} params, '
      f'{_stage1_n_trainable} trainable (must be 0).')

_rcn_core = rcn_cell._orig_mod if hasattr(rcn_cell, '_orig_mod') else rcn_cell
if hasattr(_rcn_core, 'A_dag'):
    _A_dag = _rcn_core.A_dag.detach()
    print(f'  A_dag shape={tuple(_A_dag.shape)}  norm={_A_dag.norm():.4f}  '
          f'asym={(_A_dag - _A_dag.T).abs().mean():.4f}')

print(f'  param counts : enc={sum(p.numel() for p in encoder.parameters()):,}'
      f'  rcn={sum(p.numel() for p in rcn_cell.parameters()):,}'
      f'  head={sum(p.numel() for p in regression_head.parameters()):,}'
      f'  dual_path={sum(p.numel() for p in dual_path.parameters()):,}')
print(f'[Cell 4] Stage 1 dual-path FROZEN. num_vars (for Stage 2) = {num_vars}')


# Helper used by training loop AND eval loop to compute mu_total from Stage 1.
@torch.no_grad()
def predict_mu_total(_batch):
    """Stage 1 forward (frozen): returns (mu_total, baseline_log, hr_residual).

    mu_total = mu_A + gate * mu_B(LR)   [B, 1, H, W]
    baseline_log = baseline at last step [B, 1, H, W]
    hr_residual  = HR_log1p - baseline   [B, 1, H, W]  (= sample['residual'][-1])
    """
    lr_data = _batch['lr'].to(DEVICE)
    h_init  = encoder.init_state(_batch['hetero']).to(DEVICE)
    drivers = [lr_data[t] for t in range(lr_data.shape[0])]
    seq_out = rcn_runner.run(h_init, drivers, reconstruction_sources=None)
    mu_A    = regression_head(seq_out.states[-1])
    if mu_A.dim() == 3:
        mu_A = mu_A.unsqueeze(0)
    lr_grid = batch_lr_grid_last(_batch, builder=builder, device=DEVICE)
    lr_safe = torch.nan_to_num(lr_grid, nan=0.0)
    mu_total, _mu_B, _gate = dual_path(lr_safe, mu_A)
    mu_total = torch.nan_to_num(mu_total, nan=0.0)

    bl = _batch['baseline'][-1].to(DEVICE)
    if bl.dim() == mu_total.dim() - 1:
        bl = bl.unsqueeze(0)
    bl = torch.nan_to_num(bl, nan=0.0)

    hr_res = _batch['residual'][-1].to(DEVICE)
    if hr_res.dim() == 3:
        hr_res = hr_res.unsqueeze(0)
    return mu_total, bl, hr_res


In [ ]:
# === Cell 5 : Build + init Stage 2 (warm-start OR from scratch) ===
import copy
from st_cdgm.models import CausalDiffusionDecoder
from st_cdgm.models.edm_preconditioner import EDMConfig
from omegaconf import OmegaConf as _OC

# Build UNET_KWARGS using num_vars detected in Cell 4.
UNET_KWARGS = _OC.to_container(CONFIG.diffusion.unet_kwargs, resolve=True)
for _k in ('down_block_types', 'up_block_types'):
    if _k in UNET_KWARGS and isinstance(UNET_KWARGS[_k], list):
        UNET_KWARGS[_k] = tuple(UNET_KWARGS[_k])
UNET_KWARGS['projection_class_embeddings_input_dim'] = (
    num_vars * int(CONFIG.diffusion.conditioning_dim))
print(f'[Cell 5] projection_class_embeddings_input_dim = '
      f'{UNET_KWARGS["projection_class_embeddings_input_dim"]}')

edm_cfg = EDMConfig.from_yaml_dict(CONFIG.diffusion.get('edm', {}))
_probe_sample = next(iter(val_dataset))
hr_channels = int(_probe_sample['residual'].shape[1])

diffusion_decoder = CausalDiffusionDecoder(
    in_channels=hr_channels,
    conditioning_dim=CONFIG.diffusion.conditioning_dim,
    height=int(CONFIG.diffusion.height),
    width=int(CONFIG.diffusion.width),
    unet_kwargs=UNET_KWARGS,
    scheduler_type=str(CONFIG.diffusion.scheduler_type),
    use_gradient_checkpointing=bool(CONFIG.diffusion.get('use_gradient_checkpointing', False)),
    conv_padding_mode=str(CONFIG.diffusion.get('conv_padding_mode', 'zeros')),
    anti_checkerboard=bool(CONFIG.diffusion.get('anti_checkerboard', False)),
    edm_config=edm_cfg,
    causal_concat=True,
).to(DEVICE)

# Recalibrate sigma_data to phase6 dualpath regime.
_SIGMA_DATA_CKPT = float(diffusion_decoder.edm_config.sigma_data)
diffusion_decoder.edm_config.sigma_data = float(SIGMA_DATA_NEW)
print(f'[Cell 5] sigma_data : ckpt_default={_SIGMA_DATA_CKPT:.5f} -> new={SIGMA_DATA_NEW:.5f}')

# --- Init weights ---
if FROM_SCRATCH:
    print('[Cell 5] FROM_SCRATCH=True : random weight init (no warm-start).')
else:
    print(f'[Cell 5] Warm-start from {CKPT_STAGE2_INIT}')
    if not CKPT_STAGE2_INIT.exists():
        raise FileNotFoundError(
            f'Warm-start ckpt missing : {CKPT_STAGE2_INIT}\n'
            f'Set FROM_SCRATCH=True in Cell 2 to bypass.'
        )
    _ck_init = torch.load(CKPT_STAGE2_INIT, map_location=DEVICE, weights_only=False)
    _diff_sd = _strip_prefixes(_ck_init.get('diffusion_state_dict'))
    if _diff_sd is None:
        raise RuntimeError(
            f'diffusion_state_dict absent from {CKPT_STAGE2_INIT}.'
        )
    info = diffusion_decoder.load_state_dict(_diff_sd, strict=False)
    print(f'  missing={len(info.missing_keys)}  unexpected={len(info.unexpected_keys)}')
    if info.missing_keys:
        print(f'  missing[:3] : {info.missing_keys[:3]}')
    if info.unexpected_keys:
        print(f'  unexpected[:3] : {info.unexpected_keys[:3]}')
    del _ck_init, _diff_sd

# --- EMA shadow model ---
diffusion_ema = copy.deepcopy(diffusion_decoder)
for p in diffusion_ema.parameters():
    p.requires_grad_(False)
diffusion_ema.eval()

# --- Trainable live model ---
for p in diffusion_decoder.parameters():
    p.requires_grad_(True)
diffusion_decoder.train()

_n_diff = sum(p.numel() for p in diffusion_decoder.parameters())
_n_trainable = sum(p.numel() for p in diffusion_decoder.parameters() if p.requires_grad)
print(f'[Cell 5] diffusion params : {_n_diff:,}  trainable : {_n_trainable:,}')
print(f'[Cell 5] EMA shadow params : {sum(p.numel() for p in diffusion_ema.parameters()):,}'
      f'  (decay={EMA_DECAY})')


In [ ]:
# === Cell 6 : Training loop with resume + periodic BS30 eval ===
import time
from torch.optim import Adam
from torch.optim.lr_scheduler import LinearLR

# --- Optimizer (live diffusion model only; Stage 1 frozen by Cell 4) ---
optimizer = Adam(
    [p for p in diffusion_decoder.parameters() if p.requires_grad],
    lr=LR, betas=(BETA1, BETA2), weight_decay=WEIGHT_DECAY,
)
scheduler = LinearLR(optimizer, start_factor=0.01, end_factor=1.0,
                     total_iters=WARMUP_STEPS)
scaler = torch.amp.GradScaler(enabled=(USE_AMP and DEVICE.type == 'cuda'),
                               init_scale=SCALER_INIT_SCALE)

# --- Resume ---
start_epoch = 1
best_pearson = -1.0
training_history = []

if RESUME and CKPT_LAST.exists():
    print(f'[Cell 6] RESUME from {CKPT_LAST}')
    _ck = torch.load(CKPT_LAST, map_location=DEVICE, weights_only=False)
    diffusion_decoder.load_state_dict(_strip_prefixes(_ck['diffusion_state_dict']))
    if _ck.get('diffusion_ema_state_dict') is not None:
        diffusion_ema.load_state_dict(_strip_prefixes(_ck['diffusion_ema_state_dict']))
    if _ck.get('optimizer_state_dict') is not None:
        try:
            optimizer.load_state_dict(_ck['optimizer_state_dict'])
        except Exception as _e:
            print(f'  [warn] optimizer resume failed : {_e}')
    if _ck.get('scheduler_state_dict') is not None:
        try:
            scheduler.load_state_dict(_ck['scheduler_state_dict'])
        except Exception as _e:
            print(f'  [warn] scheduler resume failed : {_e}')
    if _ck.get('scaler_state_dict') is not None:
        try:
            scaler.load_state_dict(_ck['scaler_state_dict'])
        except Exception as _e:
            print(f'  [warn] scaler resume failed : {_e}')
    start_epoch = int(_ck.get('epoch', 0)) + 1
    best_pearson = float(_ck.get('best_pearson', -1.0))
    training_history = list(_ck.get('training_history', []))
    print(f'  Resumed at epoch {start_epoch}/{EPOCHS_TARGET}, '
          f'best_pearson={best_pearson:.4f}, history={len(training_history)} entries')
    del _ck
else:
    print(f'[Cell 6] No resume : starting from epoch 1 / {EPOCHS_TARGET}')


# --- Per-micro-batch training step ---
def _autocast_ctx():
    if USE_AMP and DEVICE.type == 'cuda':
        return torch.amp.autocast(device_type='cuda', dtype=torch.float16)
    return torch.amp.autocast(device_type='cpu', enabled=False)


def train_step(_batch):
    # Stage 1 frozen forward (no_grad inside predict_mu_total).
    mu_total, baseline_log, hr_res = predict_mu_total(_batch)
    # Residual TARGET for Stage 2 = HR_log1p - baseline_log - mu_total
    #                              = (sample['residual'][-1]) - mu_total
    target_residual = hr_res - mu_total

    optimizer.zero_grad(set_to_none=True)
    with _autocast_ctx():
        loss = diffusion_decoder.compute_loss_edm(
            target=target_residual,
            conditioning=None,
            conditioning_spatial=None,
            mu_HR=mu_total,
            baseline_log=baseline_log,
            return_components=False,
        )
    if not torch.isfinite(loss):
        print(f'  [WARN] non-finite loss={loss.item()} — skipping step')
        return float('nan')

    if USE_AMP and DEVICE.type == 'cuda':
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(diffusion_decoder.parameters(), GRADIENT_CLIP)
        scaler.step(optimizer)
        scaler.update()
    else:
        loss.backward()
        torch.nn.utils.clip_grad_norm_(diffusion_decoder.parameters(), GRADIENT_CLIP)
        optimizer.step()

    scheduler.step()

    # EMA update (post-step).
    with torch.no_grad():
        for p_ema, p_live in zip(diffusion_ema.parameters(),
                                  diffusion_decoder.parameters()):
            p_ema.data.mul_(EMA_DECAY).add_(p_live.data, alpha=1.0 - EMA_DECAY)
        for b_ema, b_live in zip(diffusion_ema.buffers(),
                                  diffusion_decoder.buffers()):
            b_ema.data.copy_(b_live.data)
    return float(loss.detach().item())


# --- Per-epoch BS30 validation (uses EMA weights) ---
@torch.no_grad()
def eval_epoch():
    """Run BS30 protocol with EMA weights. Returns dict of metrics."""
    diffusion_ema.eval()
    _means, _targets, _muHR = [], [], []
    _intervention = []
    _count = 0
    _t0 = time.time()
    for _bl in iterate_batches(val_dataloader, builder, DEVICE):
        for _bb in _bl:
            if _count >= EVAL_N_BATCHES:
                break
            mu_tot, base_log, tgt = predict_mu_total(_bb)
            # K-sample ensemble with EMA weights.
            _samples = []
            for _k in range(EVAL_K_SAMPLES):
                _out = diffusion_ema.sample(
                    conditioning=None,
                    num_steps=EVAL_N_STEPS_DIFF,
                    scheduler_type='edm_karras',
                    cfg_scale=0.0,
                    apply_constraints=False,
                    mu_HR=mu_tot,
                    baseline_log=base_log,
                )
                _samples.append(_out.residual if hasattr(_out, 'residual') else _out)
            _samples = torch.stack(_samples, dim=0)
            _means.append(_samples.mean(dim=0))
            _targets.append(tgt)
            _muHR.append(mu_tot)

            if _count < EVAL_N_INTERVENTION:
                _mu_zero = torch.zeros_like(mu_tot)
                _r1 = diffusion_ema.sample(
                    conditioning=None, num_steps=EVAL_N_STEPS_DIFF,
                    scheduler_type='edm_karras', cfg_scale=0.0,
                    apply_constraints=False,
                    mu_HR=mu_tot, baseline_log=base_log,
                )
                _r0 = diffusion_ema.sample(
                    conditioning=None, num_steps=EVAL_N_STEPS_DIFF,
                    scheduler_type='edm_karras', cfg_scale=0.0,
                    apply_constraints=False,
                    mu_HR=_mu_zero, baseline_log=base_log,
                )
                _r1t = _r1.residual if hasattr(_r1, 'residual') else _r1
                _r0t = _r0.residual if hasattr(_r0, 'residual') else _r0
                _delta = (_r1t - _r0t).abs().mean().item()
                _signal = _r1t.abs().mean().item()
                _intervention.append(_delta / max(_signal, 1e-8))
            _count += 1
        if _count >= EVAL_N_BATCHES:
            break
    _eval_time = time.time() - _t0
    _pred_mean = torch.cat(_means,   dim=0).cpu()
    _targets_c = torch.cat(_targets, dim=0).cpu()
    _muHR_c    = torch.cat(_muHR,    dim=0).cpu()
    _pred_full = _muHR_c + _pred_mean
    _valid = torch.isfinite(_targets_c)
    _pc = torch.where(_valid, _pred_full, torch.zeros_like(_pred_full))
    _tc = torch.where(_valid, _targets_c, torch.zeros_like(_targets_c))
    _rmse = float(((_pc - _tc) ** 2)[_valid].mean().sqrt().item()) if _valid.any() else float('nan')

    def _pearson(a, b, eps=1e-8):
        a_c = a - a.mean(); b_c = b - b.mean()
        num = (a_c * b_c).sum()
        den = torch.sqrt((a_c * a_c).sum() * (b_c * b_c).sum() + eps)
        return float((num / den).item())

    _p_flat = _pred_full[_valid]; _t_flat = _targets_c[_valid]
    _corr_global = _pearson(_p_flat, _t_flat) if _p_flat.numel() > 1 else float('nan')

    # Per-sample mean pearson
    _per_sample = []
    for _i in range(_pred_full.shape[0]):
        _vi = _valid[_i]
        if _vi.sum() < 2: continue
        _c = _pearson(_pred_full[_i][_vi], _targets_c[_i][_vi])
        if _c == _c:
            _per_sample.append(_c)
    _corr_per_sample = float(sum(_per_sample) / max(1, len(_per_sample))) if _per_sample else float('nan')

    _dag = float(sum(_intervention) / max(1, len(_intervention))) if _intervention else float('nan')

    return {
        'pearson':           _corr_global,
        'pearson_per_sample': _corr_per_sample,
        'rmse':              _rmse,
        'mu_hr_ablation':    _dag,
        'eval_time_s':       _eval_time,
        'n_batches_eval':    len(_means),
    }


# --- Main training loop ---
print(f'\n[Cell 6] Training from epoch {start_epoch} to {EPOCHS_TARGET}'
      f'  (SMOKE_MODE={SMOKE_MODE})')

for ep in range(start_epoch, EPOCHS_TARGET + 1):
    t0 = time.time()
    diffusion_decoder.train()
    diffusion_ema.eval()

    epoch_losses = []
    _seen_batches = 0
    for batch_list in iterate_batches(train_dataloader, builder, DEVICE):
        for _micro in batch_list:
            _loss_val = train_step(_micro)
            if _loss_val == _loss_val:  # not NaN
                epoch_losses.append(_loss_val)
            _seen_batches += 1
            if _seen_batches % 30 == 0:
                _lr_now = optimizer.param_groups[0]['lr']
                print(f'  [ep{ep}/{EPOCHS_TARGET} batch {_seen_batches}] '
                      f'loss={_loss_val:.5f}  lr={_lr_now:.2e}')

    avg_loss = sum(epoch_losses) / max(1, len(epoch_losses)) if epoch_losses else float('nan')
    epoch_time = time.time() - t0
    print(f'  [ep{ep}] train done : avg_loss={avg_loss:.5f}  '
          f'time={epoch_time:.1f}s  batches={_seen_batches}')

    # Eval branch
    _record = {
        'epoch': ep,
        'train_loss': avg_loss,
        'epoch_time_s': epoch_time,
        'n_train_batches': _seen_batches,
    }
    _did_eval = False
    if ep % EVAL_EVERY_N_EPOCHS == 0:
        print(f'  [ep{ep}] Running BS30 validation (EMA weights) ...')
        _metrics = eval_epoch()
        _record.update(_metrics)
        _did_eval = True
        _p = _metrics['pearson']
        print(f'  [ep{ep}] EVAL: pearson={_p:.4f}  '
              f'per_sample={_metrics["pearson_per_sample"]:.4f}  '
              f'rmse={_metrics["rmse"]:.4f}  '
              f'mu_HR_abl={_metrics["mu_hr_ablation"] * 100:.2f}%  '
              f'eval_time={_metrics["eval_time_s"]:.1f}s')

        if _p > best_pearson:
            best_pearson = _p
            print(f'  [ep{ep}] * NEW BEST Pearson = {best_pearson:.4f}')
            _record['is_best'] = True
        else:
            _record['is_best'] = False

    training_history.append(_record)

    # Persist LAST checkpoint every epoch.
    payload = {
        'epoch': ep,
        'diffusion_state_dict': diffusion_decoder.state_dict(),
        'diffusion_ema_state_dict': diffusion_ema.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'best_pearson': best_pearson,
        'training_history': training_history,
        'sigma_data': float(SIGMA_DATA_NEW),
        'from_scratch': bool(FROM_SCRATCH),
        'smoke_mode': bool(SMOKE_MODE),
        'epochs_target': int(EPOCHS_TARGET),
    }
    torch.save(payload, CKPT_LAST)

    # Persist BEST checkpoint (by Pearson) if this epoch was eval'd and improved.
    if _did_eval and _record.get('is_best', False):
        torch.save(payload, CKPT_BEST)
        print(f'  [ep{ep}] BEST checkpoint saved -> {CKPT_BEST.name}')

    # Mirror history JSON for easy inspection.
    HISTORY_JSON.write_text(
        json.dumps(training_history, indent=2, default=str), encoding='utf-8'
    )

print(f'\n[Cell 6] Training done. epochs ran {start_epoch}..{EPOCHS_TARGET}.'
      f'  best_pearson={best_pearson:.4f}')


In [ ]:
# === Cell 7 : SMOKE TEST GO / NO-GO verdict ===
if not SMOKE_MODE:
    print('[Cell 7] SMOKE_MODE=False -> skipped (full mode, see Cell 8 for final eval).')
else:
    print('=' * 60)
    print('SMOKE TEST VERDICT (Phase 7 — Stage 2 retrain)')
    print('=' * 60)

    _trace = [(h['epoch'], h.get('pearson')) for h in training_history
              if 'pearson' in h and h['pearson'] is not None and h['pearson'] == h['pearson']]
    if not _trace:
        print('NO-GO : no eval recorded in this smoke run.')
    else:
        _peps   = [e for e, _ in _trace]
        _ppears = [p for _, p in _trace]
        print(f'Epochs evaluated   : {_peps}')
        print(f'Pearson trajectory : {[f"{x:.4f}" for x in _ppears]}')
        print(f'Best Pearson       : {best_pearson:.4f}')

        TARGET_PEARSON = 0.82
        EXPECTED_EP3   = 0.55   # rough expectation for warm-start at 3 epochs

        if best_pearson >= EXPECTED_EP3:
            if len(_ppears) >= 2:
                _delta_per_ep = (_ppears[-1] - _ppears[0]) / max(len(_ppears) - 1, 1)
            else:
                _delta_per_ep = 0.0
            _projected = _ppears[-1] + _delta_per_ep * (FULL_EPOCHS - SMOKE_EPOCHS)
            print(f'Trend slope        : {_delta_per_ep:+.4f}/epoch')
            print(f'Linear extrapolation to {FULL_EPOCHS} epochs : {_projected:.4f}')

            if _projected >= TARGET_PEARSON:
                print()
                print(f'GO : trajectory suggests reaching Pearson >= {TARGET_PEARSON}.')
                print('  Recommendation : set SMOKE_MODE = False in Cell 2,')
                print(f'                   then re-run from Cell 6 (auto-resume from {CKPT_LAST.name}).')
            else:
                print()
                print(f'MARGINAL : linear extrapolation = {_projected:.4f} < target {TARGET_PEARSON}.')
                print('  Consider : longer training, lower LR (5e-5), or warm-start')
                print('             from a different Stage 2 (e.g. non-causal checkpoint).')
        else:
            print()
            print(f'NO-GO : best_pearson={best_pearson:.4f} < expected {EXPECTED_EP3} after smoke.')
            print('  Recommendation : investigate :')
            print('   1. Train loss curve — is it actually decreasing?')
            print('   2. Stage 1 mu_total quality — print mean/std of mu_total at Cell 4.')
            print('   3. sigma_data mismatch — verify SIGMA_DATA_NEW=0.193 matches phase6.')
            print('   4. Try FROM_SCRATCH=True (cold start, slower but unbiased).')

    # mu_HR causality preservation check.
    _abl_trace = [h.get('mu_hr_ablation') for h in training_history
                  if h.get('mu_hr_ablation') is not None
                  and h.get('mu_hr_ablation') == h.get('mu_hr_ablation')]
    if _abl_trace:
        print()
        print(f'mu_HR ablation trajectory : '
              f'{[f"{x * 100:.2f}%" for x in _abl_trace]}')
        _last_abl = _abl_trace[-1]
        if _last_abl >= 0.90:
            print(f'CAUSALITY OK : mu_HR ablation {_last_abl * 100:.2f}% '
                  f'(target ~99%, Stage 2 still depends on mu_HR).')
        elif _last_abl >= 0.50:
            print(f'CAUSALITY DEGRADED : mu_HR ablation = {_last_abl * 100:.2f}%.')
        else:
            print(f'CAUSALITY LOST : mu_HR ablation = {_last_abl * 100:.2f}% — '
                  f'Stage 2 is largely ignoring mu_HR (would re-introduce non-causality).')


In [ ]:
# === Cell 8 : FULL-mode final validation (BS30) — saves final_validation_metrics.json ===
# Only runs if SMOKE_MODE=False. Mirrors phase6_dualpath_final_validation Cell 6
# format so the resulting JSON is drop-in for the 3-way comparison notebook.
if SMOKE_MODE:
    print('[Cell 8] SMOKE_MODE=True -> skipping final BS30 + JSON dump.')
else:
    import time
    from st_cdgm.evaluation import compute_f1_extremes, compute_spectrum_distance

    PHASE7_REF_DIR = RETRAIN_DIR
    PHASE7_REF_DIR.mkdir(parents=True, exist_ok=True)
    print(f'[Cell 8] Running BS30 final eval with EMA weights ...')

    # Re-use the larger BS30 protocol (same as Phase 5/6).
    N_TEST_BATCHES_FINAL = EVAL_N_BATCHES
    K_SAMPLES_FINAL      = EVAL_K_SAMPLES
    N_STEPS_DIFF_FINAL   = EVAL_N_STEPS_DIFF
    N_INTERVENTION_FINAL = EVAL_N_INTERVENTION

    diffusion_ema.eval()
    _all_means, _all_stds, _all_targets, _all_mu_HR, _intervention = [], [], [], [], []
    _t0 = time.time()
    _count = 0
    with torch.no_grad():
        for _bl in iterate_batches(val_dataloader, builder, DEVICE):
            for _bb in _bl:
                if _count >= N_TEST_BATCHES_FINAL:
                    break
                mu_tot, base_log, tgt = predict_mu_total(_bb)
                _samples = []
                for _k in range(K_SAMPLES_FINAL):
                    _out = diffusion_ema.sample(
                        conditioning=None, num_steps=N_STEPS_DIFF_FINAL,
                        scheduler_type='edm_karras', cfg_scale=0.0,
                        apply_constraints=False,
                        mu_HR=mu_tot, baseline_log=base_log,
                    )
                    _samples.append(_out.residual if hasattr(_out, 'residual') else _out)
                _samples = torch.stack(_samples, dim=0)
                _all_means.append(_samples.mean(dim=0))
                _all_stds.append(_samples.std(dim=0))
                _all_targets.append(tgt)
                _all_mu_HR.append(mu_tot)

                if _count < N_INTERVENTION_FINAL:
                    _mu_zero = torch.zeros_like(mu_tot)
                    _r1 = diffusion_ema.sample(
                        conditioning=None, num_steps=N_STEPS_DIFF_FINAL,
                        scheduler_type='edm_karras', cfg_scale=0.0,
                        apply_constraints=False, mu_HR=mu_tot, baseline_log=base_log,
                    )
                    _r0 = diffusion_ema.sample(
                        conditioning=None, num_steps=N_STEPS_DIFF_FINAL,
                        scheduler_type='edm_karras', cfg_scale=0.0,
                        apply_constraints=False, mu_HR=_mu_zero, baseline_log=base_log,
                    )
                    _r1t = _r1.residual if hasattr(_r1, 'residual') else _r1
                    _r0t = _r0.residual if hasattr(_r0, 'residual') else _r0
                    _delta = (_r1t - _r0t).abs().mean().item()
                    _signal = _r1t.abs().mean().item()
                    _intervention.append(_delta / max(_signal, 1e-8))
                    print(f'   batch {_count + 1:2d} | mu_HR ablation = '
                          f'{_intervention[-1] * 100:6.2f}%')
                else:
                    print(f'   batch {_count + 1:2d} | sampled K={K_SAMPLES_FINAL}')
                _count += 1
            if _count >= N_TEST_BATCHES_FINAL:
                break
    _eval_time = time.time() - _t0

    _pred_mean = torch.cat(_all_means,   dim=0).cpu()
    _pred_std  = torch.cat(_all_stds,    dim=0).cpu()
    _targets   = torch.cat(_all_targets, dim=0).cpu()
    _mu_HR_cat = torch.cat(_all_mu_HR,   dim=0).cpu()
    _pred_full = _mu_HR_cat + _pred_mean
    _valid = torch.isfinite(_targets)
    _pc = torch.where(_valid, _pred_full, torch.zeros_like(_pred_full))
    _tc = torch.where(_valid, _targets,   torch.zeros_like(_targets))
    _rmse  = float(((_pc - _tc) ** 2)[_valid].mean().sqrt().item()) if _valid.any() else float('nan')
    _mae   = float((_pc - _tc).abs()[_valid].mean().item()) if _valid.any() else float('nan')
    _spread = float(_pred_std[_valid].mean().item()) if _valid.any() else float('nan')

    def _pearson(a, b, eps=1e-8):
        a_c = a - a.mean(); b_c = b - b.mean()
        num = (a_c * b_c).sum()
        den = torch.sqrt((a_c * a_c).sum() * (b_c * b_c).sum() + eps)
        return float((num / den).item())

    _corr_global = float('nan'); _per_sample = []
    try:
        _p_flat = _pred_full[_valid]; _t_flat = _targets[_valid]
        if _p_flat.numel() > 1:
            _corr_global = _pearson(_p_flat, _t_flat)
        for _i in range(_pred_full.shape[0]):
            _vi = _valid[_i]
            if _vi.sum() < 2: continue
            _c = _pearson(_pred_full[_i][_vi], _targets[_i][_vi])
            if _c == _c: _per_sample.append(_c)
    except Exception as _e:
        print(f'[warn] Pearson failed : {_e}')
    _corr_per_sample = float(sum(_per_sample) / max(1, len(_per_sample))) if _per_sample else float('nan')

    _f1 = {}
    try:
        _f1 = compute_f1_extremes(_pc, _tc, threshold_percentiles=[95.0, 99.0])
    except Exception as _e:
        print(f'[warn] F1 extremes failed : {_e}')

    _rapsd_d = None
    try:
        _rapsd_d = float(compute_spectrum_distance(_pc[0], _tc[0]))
    except Exception as _e:
        print(f'[warn] RAPSD distance failed : {_e}')

    _dag_avg = float(sum(_intervention) / max(1, len(_intervention))) if _intervention else None
    if _dag_avg is None:
        _verdict_dag = 'N/A'
    elif _dag_avg < 0.001:
        _verdict_dag = 'MU_HR_IGNORED'
    elif _dag_avg < 0.01:
        _verdict_dag = 'WEAK'
    else:
        _verdict_dag = 'MU_HR_CONDITIONS'

    print()
    print('=' * 72)
    print('FINAL VALIDATION METRICS (BS30 protocol, phase7_stage2_retrain)')
    print('=' * 72)
    print(f'  N batches             : {len(_all_targets)}')
    print(f'  K samples / batch     : {K_SAMPLES_FINAL}')
    print(f'  Eval time             : {_eval_time:.1f}s')
    print(f'  RMSE (ensemble mean)  : {_rmse:.6f}')
    print(f'  MAE                   : {_mae:.6f}')
    print(f'  Spread (ensemble std) : {_spread:.6f}')
    print(f'  Pearson (global)      : {_corr_global:.4f}')
    print(f'  Pearson (per-sample)  : {_corr_per_sample:.4f}  (n={len(_per_sample)})')
    for _k, _v in _f1.items():
        print(f'  {_k:<22}: {_v:.4f}')
    if _rapsd_d is not None:
        print(f'  RAPSD distance        : {_rapsd_d:.6f}')
    if _dag_avg is not None:
        print(f'  mu_HR ablation        : {_dag_avg * 100:.3f}%  -> {_verdict_dag}')

    _metrics = {
        'checkpoint':       str(CKPT_BEST if CKPT_BEST.exists() else CKPT_LAST),
        'mode':             'phase7_stage2_retrain',
        'causal_concat':    True,
        'stage1_frozen':    True,
        'stage2_warmstart': not FROM_SCRATCH,
        'n_test_batches':   len(_all_targets),
        'k_samples':        K_SAMPLES_FINAL,
        'metrics_scope':    'full_prediction (mu_HR + delta_hat)',
        'eval_time_s':      _eval_time,
        'rmse':             _rmse,
        'mae':              _mae,
        'spread_mean':      _spread,
        'f1_extremes':      _f1,
        'pearson_corr': {
            'global':         _corr_global,
            'per_sample_avg': _corr_per_sample,
            'per_sample_n':   len(_per_sample),
            'per_sample_list': _per_sample[:64],
        },
        'rapsd_distance':   _rapsd_d,
        'mu_HR_ablation': {
            'delta_signal_ratio_avg': _dag_avg,
            'per_batch':              _intervention,
            'verdict':                _verdict_dag,
        },
        'eval_protocol': {
            'n_test_batches': N_TEST_BATCHES_FINAL,
            'k_samples':      K_SAMPLES_FINAL,
            'n_steps_diff':   N_STEPS_DIFF_FINAL,
            'scheduler':      'edm_karras',
            'cfg_scale':      0.0,
        },
        'sigma_data':       float(SIGMA_DATA_NEW),
        'training_epochs_done': int(start_epoch + len(training_history) - 1)
                                if training_history else 0,
        'best_pearson_during_training': best_pearson,
    }
    _fv_path = PHASE7_REF_DIR / 'final_validation_metrics.json'
    _fv_path.write_text(json.dumps(_metrics, indent=2, default=str), encoding='utf-8')
    print(f'\n[Cell 8] Saved : {_fv_path}')

    _N_dump = min(8, int(_targets.shape[0]))
    _payload = dict(
        target    = _targets[:_N_dump].numpy(),
        pred_full = _pred_full[:_N_dump].numpy(),
        pred_std  = _pred_std[:_N_dump].numpy(),
        valid_mask= _valid[:_N_dump].numpy().astype('float32'),
        mu_HR     = _mu_HR_cat[:_N_dump].numpy(),
        run_variant = np.array('phase7_stage2_retrain'),
    )
    _npz_path = PHASE7_REF_DIR / 'eval_samples.npz'
    np.savez_compressed(_npz_path, **_payload)
    print(f'[Cell 8] Saved : {_npz_path}  (N={_N_dump})')
